In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q numpy

In [ ]:
import os

files_to_check = [
    "/content/drive/MyDrive/baseline1_results.json",
    "/content/drive/MyDrive/baseline2_results.json",
    "/content/drive/MyDrive/rag_demo_results.json",
]

for f in files_to_check:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"✅ {f} ({size_kb:.1f} KB)")
    else:
        print(f"❌ MISSING: {f}")

✅ /content/drive/MyDrive/baseline1_results.json (317.2 KB)
✅ /content/drive/MyDrive/baseline2_results.json (324.1 KB)
✅ /content/drive/MyDrive/rag_demo_results.json (28.5 KB)


In [ ]:
import json

with open("/content/drive/MyDrive/baseline1_results.json") as f:
    data = json.load(f)

print(f"Number of entries: {len(data)}")
print(f"\nFirst entry keys: {list(data[0].keys())}")
print(f"\nExample entry:\n{json.dumps(data[0], indent=2)}")

Number of entries: 500

First entry keys: ['question', 'gold', 'aliases', 'prediction']

Example entry:
{
  "question": "What was first worn by British soldiers in India in 1845?",
  "gold": "Khaki",
  "aliases": [
    "Khaki",
    "Kackeys",
    "Kackey pants",
    "Khakis",
    "Khaki (colour)",
    "Dark khaki",
    "Khaki (color)",
    "Cackeys"
  ],
  "prediction": "Khaki Uniforms"
}


In [ ]:
import re
import string
from collections import Counter


def normalize_answer(s):
    """
    SQuAD-style answer normalization.
    Converts to lowercase, removes articles, punctuation, and extra whitespace.
    This is the standard used across open-domain QA benchmarks.
    """
    if not s:
        return ""
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))


def get_tokens(s):
    """Tokenize by whitespace after normalization."""
    if not s:
        return []
    return normalize_answer(s).split()


def exact_match_score(prediction, ground_truth):
    """Strict exact match after normalization."""
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def substring_match_score(prediction, ground_truth):
    """
    Loose match: ground truth appears as a substring of the prediction.
    Useful when the model gives verbose answers containing the correct span.
    """
    gt_norm = normalize_answer(ground_truth)
    pred_norm = normalize_answer(prediction)
    if not gt_norm:
        return 0
    return int(gt_norm in pred_norm)


def f1_score(prediction, ground_truth):
    """Token-level F1 between prediction and ground truth, SQuAD-style."""
    pred_tokens = get_tokens(prediction)
    gt_tokens = get_tokens(ground_truth)
    if not pred_tokens or not gt_tokens:
        return int(pred_tokens == gt_tokens)
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def metric_max_over_candidates(metric_fn, prediction, candidates):
    """
    TriviaQA allows multiple correct aliases (e.g., "JFK" or "John F. Kennedy").
    Return the best score against any candidate.
    """
    if not candidates:
        return 0.0
    return max(metric_fn(prediction, c) for c in candidates if c)


def evaluate_results(results):
    """
    Compute aggregate metrics over a list of result dicts.
    Each dict must have: 'prediction', 'gold', 'aliases'.
    """
    n = len(results)
    em_total = 0
    substring_total = 0
    f1_total = 0.0

    for r in results:
        candidates = [r["gold"]] + list(r.get("aliases", []))
        em_total += metric_max_over_candidates(exact_match_score, r["prediction"], candidates)
        substring_total += metric_max_over_candidates(substring_match_score, r["prediction"], candidates)
        f1_total += metric_max_over_candidates(f1_score, r["prediction"], candidates)

    return {
        "n": n,
        "exact_match": em_total / n,
        "substring_match": substring_total / n,
        "f1": f1_total / n,
    }


print("Evaluation functions defined")

Evaluation functions defined


In [ ]:
# Test cases: (prediction, gold, expected_exact, expected_substring)
tests = [
    ("Paris", "Paris", 1, 1),                                # exact hit
    ("paris", "Paris", 1, 1),                                # case insensitive
    ("The capital is Paris", "Paris", 0, 1),                 # verbose → substring yes, exact no
    ("London", "Paris", 0, 0),                               # wrong
    ("William Shakespeare", "Shakespeare", 0, 1),            # partial name
    ("Shakespeare wrote Hamlet.", "Shakespeare", 0, 1),      # with punctuation
    ("", "Paris", 0, 0),                                     # empty prediction
]

print("Running unit tests...\n")
for pred, gold, exp_em, exp_sub in tests:
    em = exact_match_score(pred, gold)
    sub = substring_match_score(pred, gold)
    f1 = f1_score(pred, gold)
    ok = "✅" if (em == exp_em and sub == exp_sub) else "❌"
    print(f"{ok} pred={pred!r:35} gold={gold!r:20} EM={em} Sub={sub} F1={f1:.2f}")

Running unit tests...

✅ pred='Paris'                             gold='Paris'              EM=1 Sub=1 F1=1.00
✅ pred='paris'                             gold='Paris'              EM=1 Sub=1 F1=1.00
✅ pred='The capital is Paris'              gold='Paris'              EM=0 Sub=1 F1=0.50
✅ pred='London'                            gold='Paris'              EM=0 Sub=0 F1=0.00
✅ pred='William Shakespeare'               gold='Shakespeare'        EM=0 Sub=1 F1=0.67
✅ pred='Shakespeare wrote Hamlet.'         gold='Shakespeare'        EM=0 Sub=1 F1=0.50
✅ pred=''                                  gold='Paris'              EM=0 Sub=0 F1=0.00


In [ ]:
CONFIGS = [
    ("Baseline 1 (zero-shot)",    "/content/drive/MyDrive/baseline1_results.json"),
    ("Baseline 2 (few-shot)",     "/content/drive/MyDrive/baseline2_results.json"),
    ("RAG (no fine-tune)",        "/content/drive/MyDrive/rag_demo_results.json"),
]

all_metrics = {}

print("="*70)
print(f"{'Configuration':<28} {'N':>6} {'EM':>8} {'Sub':>8} {'F1':>8}")
print("="*70)

for name, path in CONFIGS:
    with open(path) as f:
        data = json.load(f)
    m = evaluate_results(data)
    all_metrics[name] = m
    print(f"{name:<28} {m['n']:>6} {m['exact_match']:>8.3f} {m['substring_match']:>8.3f} {m['f1']:>8.3f}")

print("="*70)
print("\nNotes:")
print("  EM  = SQuAD-style Exact Match (normalized prediction == normalized gold/alias)")
print("  Sub = Substring Match (normalized gold/alias appears in normalized prediction)")
print("  F1  = Token-level F1 score")

Configuration                     N       EM      Sub       F1
Baseline 1 (zero-shot)          500    0.356    0.744    0.492
Baseline 2 (few-shot)           500    0.290    0.726    0.426
RAG (no fine-tune)               30    0.000    0.600    0.116

Notes:
  EM  = SQuAD-style Exact Match (normalized prediction == normalized gold/alias)
  Sub = Substring Match (normalized gold/alias appears in normalized prediction)
  F1  = Token-level F1 score


In [ ]:
print("="*70)
print("QUALITATIVE ANALYSIS: Where do the configurations disagree?")
print("="*70)

with open("/content/drive/MyDrive/baseline1_results.json") as f:
    b1_data = json.load(f)
with open("/content/drive/MyDrive/rag_demo_results.json") as f:
    rag_data = json.load(f)

# Match by question (RAG used first 30 of same 500)
rag_questions = {r["question"]: r for r in rag_data}

disagreements = []
for b1 in b1_data[:30]:  # first 30 since that's RAG's slice
    rag = rag_questions.get(b1["question"])
    if not rag:
        continue
    candidates = [b1["gold"]] + list(b1["aliases"])
    b1_correct = metric_max_over_candidates(substring_match_score, b1["prediction"], candidates)
    rag_correct = metric_max_over_candidates(substring_match_score, rag["prediction"], candidates)
    if b1_correct != rag_correct:
        disagreements.append({
            "question": b1["question"],
            "gold": b1["gold"],
            "b1_pred": b1["prediction"][:100],
            "rag_pred": rag["prediction"][:100],
            "b1_correct": bool(b1_correct),
            "rag_correct": bool(rag_correct),
        })

print(f"\nFound {len(disagreements)} questions where Baseline 1 and RAG disagree (out of 30 overlap).\n")

for d in disagreements[:8]:
    print(f"Q: {d['question'][:90]}")
    print(f"  Gold: {d['gold']}")
    print(f"  B1 ({'✅' if d['b1_correct'] else '❌'}): {d['b1_pred']}")
    print(f"  RAG ({'✅' if d['rag_correct'] else '❌'}): {d['rag_pred']}")
    print()

QUALITATIVE ANALYSIS: Where do the configurations disagree?

Found 7 questions where Baseline 1 and RAG disagree (out of 30 overlap).

Q: What was first worn by British soldiers in India in 1845?
  Gold: Khaki
  B1 (✅): Khaki Uniforms
  RAG (❌): Ford, who is not a British soldier but the founder of the Ford Motor Company in 1891, is depicted in

Q: What is the International Vehicle Registration for Cambodia?
  Gold: K
  B1 (✅): KH-XXXX or CD-XXXX (where X represents numbers) is the format for Cambodian vehicle registration pla
  RAG (❌): I'm unable to find the information about the International Vehicle Registration for Cambodia in the 

Q: What was James Herbert’s first book, published in 1974?
  Gold: Rats
  B1 (✅): The Rats.
  RAG (❌): The question appears to contain an error in the name. The correct name is Herbert Hoover. Hoover's f

Q: What is the name of the lake formed in 1932 when the Zuider Zee was cut off from the North
  Gold: Ijsselmeer
  B1 (✅): IJsselmeer
  RAG (❌): The 

In [ ]:
evaluate_code = '''"""
evaluate.py — SQuAD-style evaluation metrics for RAG pipeline
Author: Adwait Gaur
DATA 612 Group Project

Implements the canonical exact-match, substring-match, and F1 metrics
used throughout open-domain QA benchmarks. Supports multi-alias matching
required by TriviaQA.
"""

import json
import re
import string
from collections import Counter


def normalize_answer(s):
    """SQuAD-style normalization: lowercase, strip articles/punctuation/whitespace."""
    if not s:
        return ""
    def remove_articles(text):
        return re.sub(r"\\b(a|an|the)\\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(s.lower())))


def get_tokens(s):
    if not s:
        return []
    return normalize_answer(s).split()


def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def substring_match_score(prediction, ground_truth):
    gt = normalize_answer(ground_truth)
    return int(gt in normalize_answer(prediction)) if gt else 0


def f1_score(prediction, ground_truth):
    pred_tokens = get_tokens(prediction)
    gt_tokens = get_tokens(ground_truth)
    if not pred_tokens or not gt_tokens:
        return int(pred_tokens == gt_tokens)
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def metric_max_over_candidates(metric_fn, prediction, candidates):
    if not candidates:
        return 0.0
    return max(metric_fn(prediction, c) for c in candidates if c)


def evaluate_results(results):
    """Compute aggregate metrics. Each result needs 'prediction', 'gold', 'aliases'."""
    n = len(results)
    em_total = 0
    substring_total = 0
    f1_total = 0.0
    for r in results:
        candidates = [r["gold"]] + list(r.get("aliases", []))
        em_total += metric_max_over_candidates(exact_match_score, r["prediction"], candidates)
        substring_total += metric_max_over_candidates(substring_match_score, r["prediction"], candidates)
        f1_total += metric_max_over_candidates(f1_score, r["prediction"], candidates)
    return {
        "n": n,
        "exact_match": em_total / n,
        "substring_match": substring_total / n,
        "f1": f1_total / n,
    }


if __name__ == "__main__":
    import sys
    for path in sys.argv[1:]:
        with open(path) as f:
            data = json.load(f)
        m = evaluate_results(data)
        print(f"{path}: N={m[\\"n\\"]}  EM={m[\\"exact_match\\"]:.3f}  Sub={m[\\"substring_match\\"]:.3f}  F1={m[\\"f1\\"]:.3f}")
'''

with open('/content/drive/MyDrive/evaluate.py', 'w') as f:
    f.write(evaluate_code)

print("Saved evaluate.py to Google Drive")

Saved evaluate.py to Google Drive


In [ ]:
import json

with open("/content/drive/MyDrive/final_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

print("Saved final_metrics.json to Drive")
print("\nContent:")
print(json.dumps(all_metrics, indent=2))

Saved final_metrics.json to Drive

Content:
{
  "Baseline 1 (zero-shot)": {
    "n": 500,
    "exact_match": 0.356,
    "substring_match": 0.744,
    "f1": 0.49190076248927506
  },
  "Baseline 2 (few-shot)": {
    "n": 500,
    "exact_match": 0.29,
    "substring_match": 0.726,
    "f1": 0.4264568783408919
  },
  "RAG (no fine-tune)": {
    "n": 30,
    "exact_match": 0.0,
    "substring_match": 0.6,
    "f1": 0.1156431084288013
  }
}
